# Base dos Dados + BigQuery: Engenharia de Dados

## O que é a Base dos Dados?

A [Base dos Dados](https://basedosdados.org) é uma plataforma brasileira de dados públicos que disponibiliza **datasets tratados e prontos para análise** via Google BigQuery (gratuito para consultas públicas).

### Dados disponíveis:
- IBGE (Censo, PNAD, PIB)
- TSE (eleições, candidatos)
- INEP (ENEM, IDEB)
- ANS (saúde suplementar)
- Tesouro Nacional (orçamento público)
- E muito mais...

---

## Como Acessar

### Passo 1 — Criar conta Google e projeto no BigQuery
1. Acesse [console.cloud.google.com](https://console.cloud.google.com)
2. Crie um projeto (ex: `meu-projeto-bd`)
3. Ative a **BigQuery API**

### Passo 2 — Criar credencial de serviço
1. No console GCP: **APIs & Services → Credentials → Create Credentials → Service Account**
2. Baixe o arquivo JSON da credencial
3. Guarde o caminho do arquivo JSON

### Passo 3 — Instalar bibliotecas
```bash
pip install basedosdados google-cloud-bigquery pandas db-dtypes
```

### Passo 4 — Explorar datasets
Acesse [basedosdados.org/search](https://basedosdados.org/search) e filtre por tema, estado ou palavra-chave.

In [7]:
# Instalação das dependências (execute uma vez)
# !pip install basedosdados google-cloud-bigquery pandas db-dtypes plotly

---
## 1. Configuração da Autenticação

In [8]:
import os
import pandas as pd
from google.cloud import bigquery
from google.oauth2 import service_account

# Opção A: via variável de ambiente (recomendado)
# os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "caminho/para/credencial.json"

# Opção B: via service account explícita
# CREDENCIAL_PATH = "caminho/para/credencial.json"
# credentials = service_account.Credentials.from_service_account_file(CREDENCIAL_PATH)

PROJECT_ID = "seu-projeto-gcp"  # <- substitua pelo seu project ID

# Inicializa o cliente BigQuery
# client = bigquery.Client(project=PROJECT_ID, credentials=credentials)
client = bigquery.Client(project=PROJECT_ID)  # usa GOOGLE_APPLICATION_CREDENTIALS

print(f"Cliente BigQuery configurado para o projeto: {PROJECT_ID}")

ModuleNotFoundError: No module named 'google'

---
## 2. Primeira Consulta — Explorando o Catálogo BD

Todo dado da Base dos Dados está no dataset público `basedosdados` no BigQuery.

**Formato do endereço:** `basedosdados.<dataset>.<tabela>`

In [ ]:
# Função auxiliar para executar queries
def query_bd(sql: str, project: str = PROJECT_ID) -> pd.DataFrame:
    """Executa uma query no BigQuery e retorna um DataFrame."""
    job_config = bigquery.QueryJobConfig()
    query_job = client.query(sql, job_config=job_config)
    df = query_job.to_dataframe()
    print(f"Linhas retornadas: {len(df):,} | Colunas: {list(df.columns)}")
    return df

# Consulta simples: PIB Municipal do IBGE
sql_pib = """
SELECT
    ano,
    id_municipio,
    sigla_uf,
    pib,
    impostos_liquidos,
    va_agropecuaria,
    va_industria,
    va_servicos
FROM `basedosdados.br_ibge_pib.municipio`
WHERE ano = 2020
  AND sigla_uf = 'RJ'
ORDER BY pib DESC
LIMIT 10
"""

df_pib = query_bd(sql_pib)
df_pib.head(10)

---
## 3. Usando a Biblioteca `basedosdados` (modo simplificado)

In [ ]:
import basedosdados as bd

# Configura o projeto (necessário na primeira vez)
# bd.config.project_id = PROJECT_ID

# Leitura direta com a lib basedosdados
df_bd = bd.read_sql(
    query="""
    SELECT *
    FROM `basedosdados.br_ibge_pib.municipio`
    WHERE ano = 2020 AND sigla_uf = 'SP'
    LIMIT 20
    """,
    billing_project_id=PROJECT_ID
)

df_bd.head()

---
## 4. Engenharia de Dados — Pipeline Completo

Vamos construir um pipeline ETL (Extract → Transform → Load) com dados reais do IBGE.

### 4.1 Extract — Coleta de Dados Brutos

In [ ]:
# Extração: PIB municipal histórico (2010-2020)
sql_extract = """
SELECT
    p.ano,
    p.id_municipio,
    p.sigla_uf,
    p.pib,
    p.va_agropecuaria,
    p.va_industria,
    p.va_servicos,
    p.impostos_liquidos,
    p.populacao
FROM `basedosdados.br_ibge_pib.municipio` p
WHERE p.ano BETWEEN 2010 AND 2020
  AND p.sigla_uf IN ('RJ', 'SP', 'MG', 'RS', 'BA')
ORDER BY p.ano, p.pib DESC
"""

df_raw = query_bd(sql_extract)
print("\nShape dos dados brutos:", df_raw.shape)
df_raw.head()

### 4.2 Transform — Limpeza e Enriquecimento

In [ ]:
import numpy as np

def transformar_pib(df: pd.DataFrame) -> pd.DataFrame:
    """Aplica transformações no dataset de PIB municipal."""
    df = df.copy()
    
    # 1. Remover nulos
    df = df.dropna(subset=["pib", "populacao"])
    
    # 2. Garantir tipos corretos
    df["ano"] = df["ano"].astype(int)
    df["pib"] = pd.to_numeric(df["pib"], errors="coerce")
    df["populacao"] = pd.to_numeric(df["populacao"], errors="coerce")
    
    # 3. Feature engineering: PIB per capita
    df["pib_per_capita"] = df["pib"] / df["populacao"]
    
    # 4. Composição setorial (%)
    df["pct_agro"]     = (df["va_agropecuaria"] / df["pib"]) * 100
    df["pct_industria"] = (df["va_industria"]    / df["pib"]) * 100
    df["pct_servicos"]  = (df["va_servicos"]     / df["pib"]) * 100
    df["pct_impostos"]  = (df["impostos_liquidos"] / df["pib"]) * 100
    
    # 5. Classificação por porte econômico
    bins   = [0, 1e6, 1e7, 1e8, 1e9, np.inf]
    labels = ["Micro", "Pequeno", "Médio", "Grande", "Metrópole"]
    df["porte_economico"] = pd.cut(df["pib"], bins=bins, labels=labels)
    
    # 6. Normalização do PIB (z-score por ano)
    df["pib_zscore"] = df.groupby("ano")["pib"].transform(
        lambda x: (x - x.mean()) / x.std()
    )
    
    return df

df_transformed = transformar_pib(df_raw)
print("Shape após transformação:", df_transformed.shape)
print("\nNovas colunas:", [c for c in df_transformed.columns if c not in df_raw.columns])
df_transformed.head()

### 4.3 Validação de Qualidade dos Dados

In [6]:
def validar_dados(df: pd.DataFrame) -> dict:
    """Gera relatório de qualidade do dataset."""
    relatorio = {
        "total_linhas": len(df),
        "total_colunas": len(df.columns),
        "nulos_por_coluna": df.isnull().sum().to_dict(),
        "pct_nulos": (df.isnull().sum() / len(df) * 100).round(2).to_dict(),
        "duplicatas": df.duplicated().sum(),
        "anos_cobertos": sorted(df["ano"].unique().tolist()),
        "ufs_cobertas": sorted(df["sigla_uf"].unique().tolist()),
        "municipios_distintos": df["id_municipio"].nunique(),
    }
    return relatorio

relatorio = validar_dados(df_transformed)

print("=== RELATÓRIO DE QUALIDADE ===")
print(f"Total de linhas    : {relatorio['total_linhas']:,}")
print(f"Total de colunas   : {relatorio['total_colunas']}")
print(f"Duplicatas         : {relatorio['duplicatas']}")
print(f"Municípios únicos  : {relatorio['municipios_distintos']}")
print(f"Anos cobertos      : {relatorio['anos_cobertos']}")
print(f"UFs cobertas       : {relatorio['ufs_cobertas']}")
print("\nNulos por coluna:")
for col, n in relatorio["nulos_por_coluna"].items():
    if n > 0:
        print(f"  {col}: {n} ({relatorio['pct_nulos'][col]}%)")

NameError: name 'df_transformed' is not defined

---
## 5. Análise Exploratória (EDA)

In [ ]:
# Estatísticas descritivas
df_transformed[["pib", "pib_per_capita", "pct_industria", "pct_servicos"]].describe().round(2)

In [ ]:
# Ranking de municípios por PIB per capita em 2020
top10_per_capita = (
    df_transformed[df_transformed["ano"] == 2020]
    .nlargest(10, "pib_per_capita")[["id_municipio", "sigla_uf", "pib", "pib_per_capita", "porte_economico"]]
    .reset_index(drop=True)
)

print("Top 10 municípios por PIB per capita (2020)")
top10_per_capita

In [ ]:
# Evolução do PIB total por UF ao longo dos anos
pib_uf_ano = (
    df_transformed
    .groupby(["ano", "sigla_uf"])["pib"]
    .sum()
    .reset_index()
    .rename(columns={"pib": "pib_total"})
)

# Pivot para visualização tabular
pib_pivot = pib_uf_ano.pivot(index="ano", columns="sigla_uf", values="pib_total")
print("PIB total por UF (em R$ mil):")
pib_pivot.applymap(lambda x: f"{x/1e9:.1f}B")  # em bilhões

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gráfico 1: Evolução do PIB por UF
for uf in pib_uf_ano["sigla_uf"].unique():
    dados_uf = pib_uf_ano[pib_uf_ano["sigla_uf"] == uf]
    axes[0].plot(dados_uf["ano"], dados_uf["pib_total"] / 1e9, marker="o", label=uf)

axes[0].set_title("Evolução do PIB Total por UF")
axes[0].set_xlabel("Ano")
axes[0].set_ylabel("PIB (R$ bilhões)")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Gráfico 2: Composição setorial média (2020)
composicao_2020 = (
    df_transformed[df_transformed["ano"] == 2020]
    .groupby("sigla_uf")[["pct_agro", "pct_industria", "pct_servicos"]]
    .mean()
    .round(1)
)

composicao_2020.plot(kind="bar", ax=axes[1], colormap="Set2")
axes[1].set_title("Composição Setorial Média por UF (2020)")
axes[1].set_xlabel("UF")
axes[1].set_ylabel("% do PIB")
axes[1].legend(["Agropecuária", "Indústria", "Serviços"])
axes[1].tick_params(axis="x", rotation=0)
axes[1].grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.show()

---
## 6. Agregações Avançadas com BigQuery SQL

In [ ]:
# Window functions: crescimento anual do PIB por município
sql_window = """
WITH pib_base AS (
    SELECT
        ano,
        id_municipio,
        sigla_uf,
        pib,
        LAG(pib) OVER (PARTITION BY id_municipio ORDER BY ano) AS pib_ano_anterior
    FROM `basedosdados.br_ibge_pib.municipio`
    WHERE sigla_uf = 'RJ'
      AND ano BETWEEN 2015 AND 2020
)
SELECT
    ano,
    id_municipio,
    sigla_uf,
    ROUND(pib, 2)                                               AS pib,
    ROUND(pib_ano_anterior, 2)                                  AS pib_ano_anterior,
    ROUND((pib - pib_ano_anterior) / pib_ano_anterior * 100, 2) AS crescimento_pct,
    RANK() OVER (PARTITION BY ano ORDER BY pib DESC)            AS ranking_pib_rj
FROM pib_base
WHERE pib_ano_anterior IS NOT NULL
ORDER BY ano DESC, pib DESC
LIMIT 20
"""

df_window = query_bd(sql_window)
df_window.head(15)

In [ ]:
# Cruzamento: PIB + dados eleitorais do TSE
sql_join = """
SELECT
    p.ano,
    p.sigla_uf,
    ROUND(SUM(p.pib) / 1e9, 2)           AS pib_total_bilhoes,
    ROUND(AVG(p.pib / p.populacao), 2)   AS pib_pc_medio,
    COUNT(DISTINCT p.id_municipio)        AS qtd_municipios,
    SUM(p.populacao)                      AS populacao_total
FROM `basedosdados.br_ibge_pib.municipio` p
WHERE p.ano = 2020
  AND p.populacao IS NOT NULL
GROUP BY p.ano, p.sigla_uf
ORDER BY pib_total_bilhoes DESC
LIMIT 15
"""

df_uf_summary = query_bd(sql_join)
df_uf_summary

---
## 7. Load — Salvando os Dados Processados

In [ ]:
# 7.1 Exportar para CSV
OUTPUT_CSV = "pib_municipal_tratado.csv"
df_transformed.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig", sep=";")
print(f"Dados exportados para: {OUTPUT_CSV}")

# 7.2 Exportar para Parquet (formato colunar eficiente)
OUTPUT_PARQUET = "pib_municipal_tratado.parquet"
df_transformed.to_parquet(OUTPUT_PARQUET, index=False, engine="pyarrow")
print(f"Dados exportados para: {OUTPUT_PARQUET}")

# 7.3 Verificar tamanhos
import os
for f in [OUTPUT_CSV, OUTPUT_PARQUET]:
    size_kb = os.path.getsize(f) / 1024
    print(f"  {f}: {size_kb:.1f} KB")

In [ ]:
# 7.4 Opcional: fazer upload de volta ao BigQuery (sua tabela própria)
def upload_to_bigquery(df: pd.DataFrame, dataset_id: str, table_id: str):
    """Carrega um DataFrame para uma tabela no BigQuery."""
    table_ref = f"{PROJECT_ID}.{dataset_id}.{table_id}"
    job_config = bigquery.LoadJobConfig(
        write_disposition="WRITE_TRUNCATE",  # substitui a tabela
        autodetect=True
    )
    job = client.load_table_from_dataframe(df, table_ref, job_config=job_config)
    job.result()  # aguarda conclusão
    print(f"Tabela criada/atualizada: {table_ref}")
    print(f"Linhas carregadas: {len(df):,}")

# Exemplo de uso:
# upload_to_bigquery(df_transformed, "meu_dataset", "pib_municipal_enriquecido")

---
## 8. Outros Datasets Úteis da Base dos Dados

| Dataset | Tabela BigQuery | Descrição |
|---|---|---|
| ENEM | `basedosdados.br_inep_enem.microdados` | Microdados do ENEM por aluno |
| Eleições | `basedosdados.br_tse_eleicoes.candidatos` | Candidatos por eleição |
| PNAD Contínua | `basedosdados.br_ibge_pnadc.microdados` | Emprego e renda |
| Saúde/ANS | `basedosdados.br_ans_beneficiarios.microdados` | Beneficiários de planos |
| Siconfi | `basedosdados.br_me_siconfi.municipio_despesas` | Despesas municipais |
| RAIS | `basedosdados.br_me_rais.microdados_vinculos` | Mercado de trabalho formal |

**Dica:** Explore o catálogo completo em [basedosdados.org/search](https://basedosdados.org/search)

In [ ]:
# Exemplo rápido: ENEM por estado (amostra)
sql_enem = """
SELECT
    ano,
    sigla_uf_prova AS uf,
    COUNT(*)                            AS total_inscritos,
    ROUND(AVG(nota_mt), 1)              AS media_matematica,
    ROUND(AVG(nota_lc), 1)              AS media_linguagens,
    ROUND(AVG(nota_redacao), 1)         AS media_redacao
FROM `basedosdados.br_inep_enem.microdados`
WHERE ano = 2022
  AND nota_mt IS NOT NULL
GROUP BY ano, uf
ORDER BY media_matematica DESC
LIMIT 10
"""

# df_enem = query_bd(sql_enem)
# df_enem  # (descomente para executar)
print("Query ENEM pronta — descomente para executar")

---
## 9. Resumo do Pipeline ETL

```
┌─────────────────────────────────────────────────────────────┐
│                    PIPELINE ETL - BASE DOS DADOS            │
├──────────────┬──────────────────┬───────────────────────────┤
│   EXTRACT    │    TRANSFORM     │          LOAD             │
├──────────────┼──────────────────┼───────────────────────────┤
│ BigQuery SQL │ • Limpeza nulos  │ • CSV (análise simples)   │
│ basedosdados │ • Feature eng.   │ • Parquet (performance)   │
│ (datasets    │ • Classificação  │ • BigQuery (escala)       │
│  públicos)   │ • Normalização   │ • SQLite/PostgreSQL       │
│              │ • Validação      │   (aplicações locais)     │
└──────────────┴──────────────────┴───────────────────────────┘
```

### Referências
- [Base dos Dados — Documentação](https://basedosdados.org/docs)
- [Google BigQuery — Python Client](https://cloud.google.com/bigquery/docs/reference/libraries)
- [basedosdados Python SDK](https://github.com/basedosdados/basedosdados-python)